In [96]:
import torch
from at2v.anitag2vec import get_setup, SetupConfig, AniTag2VecRunner
import torch.nn.functional as F

In [97]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = SetupConfig.load_from_file("../setup_params.json")
data, tagtok, anitag2vec = get_setup(
    cfg,
    device=device,
    prefix_path= ".."
)

anitag2vec.load_state_dict(torch.load("../checkpoints/anitag2vec_e15_s50000_p1871744.pth"))
anitag2vec.to(device)
anitag2vec.eval()

Loading tokenizer from '../checkpoints/token_vocab_size_5000_freq_3.json'..
Done!


AniTag2Vec(
  (emb): Embedding(5100, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (linproj): Linear(in_features=256, out_features=128, bias=True)
)

In [98]:
runner = AniTag2VecRunner(tagtok, anitag2vec)

In [99]:
def compare(a: str, b: str):
    ax = runner.run_inference_human([a])
    bx = runner.run_inference_human([b])
    howmuch = ((F.normalize(ax) @ F.normalize(bx).T).item())
    print(f"{howmuch:.2f}: '{a}' vs '{b}'")

In [ ]:
print("Permutation invariance:")
compare("#1girl #1boy", "#1boy #1girl")
compare("#nijigasaki #rina_tennoji", "#rina_tennoji #nijigasaki")

print("\nOppositeness and likeliness:")
compare("#nikke", "#blue_archive")
compare("#hayase_yuuka", "#blue_archive")
compare("#♀", "#girl")
compare("#♂", "#boy")
compare("#girl", "#boy")
compare("#♂", "#♀")
compare("#♂", "#unrelated")

print("\nSubsetness:")
compare("#1girl", "#1boy #1girl")
compare("#1boy", "#1boy #1girl")


print("\nTrying combinations it definitely has never seen:")
compare("#A #B #C", "#C #B #A")
compare("#A #B #C", "#B #C #A")
compare("#A #B #C", "#C")
compare("#A #B #C", "#B #C")

print("\nVery unlikely combinations:")
compare("#foo #bar", "#bar #foo")
compare("#foo #bar", " #foo")

Permutation invariance:
1.00: '#1girl #1boy' vs '#1boy #1girl'
0.70: '#nijigasaki #rina_tennoji' vs '#rina_tennoji #nijigasaki'

Oppositeness and likeliness:
0.39: '#nikke_goddessofvictory' vs '#blue_archive'
0.54: '#hayase_yuuka' vs '#blue_archive'
0.83: '#♀' vs '#girl'
0.74: '#♂' vs '#boy'
0.68: '#girl' vs '#boy'
1.00: '#♂' vs '#♀'
0.63: '#♂' vs '#unrelated'

Subsetness:
0.86: '#1girl' vs '#1boy #1girl'
0.86: '#1boy' vs '#1boy #1girl'

Trying combinations it definitely has never seen:
0.63: '#A #B #C' vs '#C #B #A'
0.63: '#A #B #C' vs '#B #C #A'
0.36: '#A #B #C' vs '#C'
0.66: '#A #B #C' vs '#B #C'

Very unlikely combinations:
0.71: '#foo #bar' vs '#bar #foo'
0.92: '#foo #bar' vs ' #foo'


In [101]:
query = data.real_examples[600]
ranked = runner.rank_cosim(
    query,
    data.real_examples[2000:2500]
)

In [ ]:
print("Closest to:", ", ".join(query))
for index, (cosim, tags) in enumerate(ranked[:10]):
    perc = (1 + cosim.item()) / 2
    print(f"# {index + 1}, {round(perc * 100 * 100) / 100}%: {', '.join(sorted(tags))}")